In [0]:
%run ../config/config-env

In [0]:
from pyspark.sql import functions as F


In [0]:
bronze_table =f'{catalog_name}.{bronze_schema}.results'
silver_table =f'{catalog_name}.{silver_schema}.results'

In [0]:
results_df = spark.read.table(bronze_table)
display(results_df)

In [0]:
results_dropped_df = results_df.drop('url')

In [0]:
results_renamed_df = results_dropped_df.withColumnsRenamed({
        'constructorId':"constructor_id", "driverId":'driver_id', 
        'raceName': 'race_name', 'positionText': 'finished_position_text',
        'date': 'race_date', 'grid':'grid_position', 'laps':'completed_laps', 
        'number': 'car_number', 'position': 'finish_position'
})

In [0]:
results_dropped_df = results_renamed_df.dropna(subset=['season', 'round', 'constructor_id', 'driver_id'])

In [0]:
results_deduplicated_df = results_dropped_df.dropDuplicates()

In [0]:
results_final_df = results_deduplicated_df.withColumn('race_name', F.initcap(F.col('race_name')))

In [0]:
display(results_final_df)

In [0]:
(
    results_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
spark.read.table(silver_table).display()

In [0]:
display(results_final_df.count() - results_df.count())